In [1]:
from neo4j import GraphDatabase
import networkx as nx

# === Neo4j 连接 ===
uri = "bolt://localhost:7687"
username = "neo4j"
password = "your_password"

driver = GraphDatabase.driver(uri, auth=(username, password))


def load_graph_undirected(driver):
    """
    从 Neo4j 读取图，构建 NetworkX 无向图
    """
    G = nx.Graph()
    with driver.session() as session:
        result = session.run("""
        MATCH (a)-[r]->(b)
        RETURN id(a) AS src, id(b) AS tgt, type(r) AS rel
        """)
        for record in result:
            G.add_edge(
                record["src"],
                record["tgt"],
                relation=record["rel"]
            )
    return G


G = load_graph_undirected(driver)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description='The query used a deprecated function: `id`.', position=<SummaryInputPosition line=3, column=24, offset=43>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'column': 24, 'offset': 43, 'line': 3}}> for query: '\n        MATCH (a)-[r]->(b)\n        RETURN id(a) AS src, id(b) AS tgt, type(r) AS rel\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description='The query used a deprecated function: `id`.', position=<SummaryInputPosition line=3, column=38, offset=57>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION

Graph loaded: 137 nodes, 200 edges


In [2]:
import community as community_louvain

# 运行 Louvain
partition = community_louvain.best_partition(G)

# partition: node_id -> community_id
print(f"Detected {len(set(partition.values()))} communities")


Detected 10 communities


In [3]:
from collections import Counter

community_sizes = Counter(partition.values())

print("Top communities by size:")
for cid, size in community_sizes.most_common(10):
    print(f"Community {cid}: {size} nodes")


Top communities by size:
Community 8: 31 nodes
Community 5: 21 nodes
Community 3: 20 nodes
Community 2: 20 nodes
Community 4: 16 nodes
Community 1: 7 nodes
Community 6: 6 nodes
Community 7: 6 nodes
Community 9: 6 nodes
Community 0: 4 nodes


In [4]:
def get_community_subgraph(G, partition, community_id):
    nodes = [n for n, c in partition.items() if c == community_id]
    return G.subgraph(nodes)


# 示例：取最大社区
largest_community = community_sizes.most_common(1)[0][0]
subG = get_community_subgraph(G, partition, largest_community)

print(
    f"Community {largest_community}:",
    subG.number_of_nodes(),
    "nodes,",
    subG.number_of_edges(),
    "edges"
)


Community 8: 31 nodes, 53 edges
